In [55]:
import yaml
import shutil
from pathlib import Path
import sys
sys.path.append('../')
from src.data.utils import (
    create_dataset_manifest, 
    validate_clips_parameters,
    get_clips_directory_name,
    get_oversample_directory_name,
    create_base_splits_manifest,
    enrich_splits_manifest,
    extract_dataset_identifier
)

In [56]:
try:
    with open("../config/params.yml", "r") as f:
        params = yaml.safe_load(f)
except FileNotFoundError:
    print("ERROR: config/params.yml no encontrado.")
    params = None
except Exception as e:
    print(f"ERROR al leer config/params.yml: {e}")
    params = None

if params:
    seed = params.get('random_seed', 42)

    exp_params = params.get('experiment', {})
    exp_name = exp_params.get('name', 'default_experiment')
    exp_base_dir = Path(exp_params.get('base_dir', '../experiments'))
    experiment_dir = exp_base_dir / exp_name

    # Directorio del exp y guardar config
    experiment_dir.mkdir(parents=True, exist_ok=True)
    config_run_path = experiment_dir / "config_run.yml"

    try:
        shutil.copyfile("../config/params.yml", config_run_path)
        print(f"Directorio del experimento creado en: {experiment_dir.resolve()}")
        print(f"Configuración de esta ejecución guardada en: {config_run_path.resolve()}")
    except Exception as e:
        print(f"ERROR al copiar params.yml: {e}")

    # Definir subdirectorios de salida del experimento
    dp_params = params.get('data_pipeline', {})
    manifests_subdir = experiment_dir / dp_params.get('manifests_subdir', 'results/tables')

    # Crear subdirectorios de resultados
    manifests_subdir.mkdir(parents=True, exist_ok=True)

    # Definir fuentes de datos
    ds_params = params.get('data_source', {})
    raw_videos_dir = Path(ds_params.get('raw_videos_dir', '../data/raw/dataset_videos_original'))
    
    # Extraer identificador del dataset automáticamente
    dataset_id = extract_dataset_identifier(raw_videos_dir)
    print(f"Dataset identificado: '{dataset_id}'")

    # Parámetros de pre procesamiento
    vp_params = params.get('video_processing', {})
    clip_length = vp_params.get('clip_length', 16)
    max_segments = vp_params.get('max_segments_per_video', 32)
    overlapping = vp_params.get('overlapping', True)
    stride = vp_params.get('stride', 8)
    balance_mode = dp_params.get('balance_mode', 'none')
    max_ratio = vp_params.get('balance_max_ratio', 1.2)
    split_ratios = vp_params.get('split_ratios', [0.64, 0.16, 0.20])
    train_r, val_r, test_r = split_ratios
    
    # Generar el nombre del directorio de clips basado en parámetros y dataset
    clips_dir_name = get_clips_directory_name(clip_length, max_segments, overlapping, stride, dataset_id)
    clips_dir = Path('../data/interim') / clips_dir_name
    
    # Generar directorio de oversample solo si el modo de balanceo lo requiere
    if balance_mode == 'oversample':
        oversample_dir_name = get_oversample_directory_name(clip_length, max_segments, overlapping, stride, max_ratio, dataset_id)
        oversample_clips_dir = Path('../data/interim') / oversample_dir_name
    else:
        oversample_clips_dir = None  # No se necesita para undersample o none
    
    print(f"\n{'='*60}")
    print(f"Configuración:")
    print(f"{'='*60}")
    print(f"Dataset identificado: '{dataset_id}'")
    print(f"Parámetros de generación de clips:")
    print(f"  - clip_length: {clip_length}")
    print(f"  - max_segments_per_video: {max_segments}")
    print(f"  - overlapping: {overlapping}")
    print(f"  - stride: {stride if overlapping else 'N/A'}")
    print(f"  - balance_mode: {balance_mode}")
    print(f"\nDirectorios generados automáticamente:")
    print(f"  Clips base: {clips_dir.resolve()}")
    if oversample_clips_dir:
        print(f"  Clips oversample: {oversample_clips_dir.resolve()}")
    print(f"{'='*60}\n")
    
    # Parámetros para estructura optimizada
    splitted_manifest_name = f"manifest_splitted_{dataset_id}.csv"
    
else:
    print("No se pudo cargar la configuración.")

Directorio del experimento creado en: D:\Dataset\experiments\exp_50
Configuración de esta ejecución guardada en: D:\Dataset\experiments\exp_50\config_run.yml
Dataset identificado: 'original'

Configuración:
Dataset identificado: 'original'
Parámetros de generación de clips:
  - clip_length: 16
  - max_segments_per_video: 32
  - overlapping: False
  - stride: N/A
  - balance_mode: undersample

Directorios generados automáticamente:
  Clips base: D:\Dataset\data\interim\clips_original_len16_seg32_uniform



## 1. Extracción de Clips

In [57]:
if not params:
    print("No se ha cargado la configuración, cargar primero el archivo params.yml.")
    raise ValueError("No se ha cargado la configuración.")

# Verificar si ya existen clips con estos parámetros
if clips_dir.exists() and any(clips_dir.iterdir()):
    print(f"Se encontró directorio de clips: {clips_dir.name}")
    
    params_match, saved_metadata = validate_clips_parameters(
        clips_dir,
        clip_length,
        max_segments,
        overlapping,
        stride
    )
    
    if params_match:
        print(f"Reutilizando clips de: {clips_dir.resolve()}")
        print("Omitiendo extracción de clips.\n")
    else:
        # Esto no debería ocurrir porque el nombre se genera basándose en los parámetros
        print(f"\nError inesperado: El directorio {clips_dir.name} existe pero con parámetros incorrectos")
        print(f"   Esto indica un problema con generation_metadata.json")
        print(f"   Solución: Eliminar {clips_dir.name} o su archivo generation_metadata.json")
        raise ValueError("Conflicto de parámetros en directorio de clips")
else:
    # No existen clips, generarlos
    print(f"\n{'='*60}")
    print(f"Generando clips en: {clips_dir.resolve()}")
    print(f"Parámetros:")
    print(f"  clip_length: {clip_length}")
    print(f"  max_segments_per_video: {max_segments}")
    print(f"  overlapping: {overlapping}")
    print(f"  stride: {stride}")
    print(f"{'='*60}\n")
    
    # Comando base
    cmd_parts = [
        "../src/data/clip_sampler.py",
        f"--input_dir {raw_videos_dir}",
        f"--output_dir {clips_dir}",
        f"--clip_length {clip_length}",
        f"--max_segments {max_segments}"
    ]
    
    # --overlapping solo si es True
    if overlapping:
        cmd_parts.append("--overlapping")
        cmd_parts.append(f"--stride {stride}")
    
    # Unir el comando
    cmd = " ".join(cmd_parts)
    %run {cmd}

print(f"\nDirectorio de clips confirmado: {clips_dir.resolve()}")

Se encontró directorio de clips: clips_original_len16_seg32_uniform
Los clips existentes fueron generados con los MISMOS parámetros
Reutilizando clips de: D:\Dataset\data\interim\clips_original_len16_seg32_uniform
Omitiendo extracción de clips.


Directorio de clips confirmado: D:\Dataset\data\interim\clips_original_len16_seg32_uniform


## 2. Generación de manifest CSV 

In [58]:
# El manifest base se nombra según el directorio de clips usado
base_manifest_csv = manifests_subdir / f"manifest_{clips_dir.name}.csv"

# Verificar si ya existe el manifest
if base_manifest_csv.exists():
    print(f"El manifest ya existe en: {base_manifest_csv.resolve()}")
    print("Omitiendo generación de manifest.")
else:
    print(f"Generando manifest desde: {clips_dir.name}")
    print(f"Guardando en: {base_manifest_csv.resolve()}")
    try:
        create_dataset_manifest(input_dir=str(clips_dir), output_csv=str(base_manifest_csv))
        print(f"Manifiesto generado en: {base_manifest_csv.resolve()}")
    except Exception as e:
        print(f"ERROR al generar el manifiesto: {e}")

manifest_for_next_step = base_manifest_csv if base_manifest_csv.exists() else None

Generando manifest desde: clips_original_len16_seg32_uniform
Guardando en: D:\Dataset\experiments\exp_50\results\tables\manifest_clips_original_len16_seg32_uniform.csv
Escaneando directorio: ..\data\interim\clips_original_len16_seg32_uniform
Archivo CSV creado con 318 registros en: ..\experiments\exp_50\results\tables\manifest_clips_original_len16_seg32_uniform.csv
  Clase 'Normal': 192 videos
  Clase 'Robbery': 126 videos
Manifiesto generado en: D:\Dataset\experiments\exp_50\results\tables\manifest_clips_original_len16_seg32_uniform.csv


## 3. Dividir el dataset en splits (train/val/test)

In [59]:
if manifest_for_next_step:

    interim_dir = clips_dir.parent
    base_splits_manifest = interim_dir / splitted_manifest_name
    
    print(f"Usando manifest: {manifest_for_next_step.name}")
    
    # Verificar si ya existe el manifest base de splits
    if base_splits_manifest.exists():
        print(f"\nManifest base de splits ya existe: {base_splits_manifest.name}")
        print(f"  Este manifest define la distribución de splits para todos los experimentos")
        print(f"  Reutilizando splits existentes para garantizar consistencia\n")
    else:
        print(f"\nGenerando manifest base de splits")
        print(f"  Este manifest se usará en todos los experimentos futuros")
        print(f"  Contiene: class, directoryname, split")
        print(f"  Guardando en: {base_splits_manifest.resolve()}\n")
        
        try:
            create_base_splits_manifest(
                input_manifest_csv=str(manifest_for_next_step),
                output_csv=str(base_splits_manifest),
                ratios=(train_r, val_r, test_r),
                random_seed=seed
            )
            print(f"\nDivisión en splits completada")
        except Exception as e:
            print(f"ERROR al crear el manifest base de splits: {e}")
            base_splits_manifest = None
    
    # Enriquecer el manifest base con información específica de este experimento
    # Esto agrega: directorypath y numclips (que varían según configuración)
    if base_splits_manifest and base_splits_manifest.exists():
        experiment_splits_manifest = manifests_subdir / f"manifest_splits_{clips_dir.name}.csv"
        
        print(f"  Agregando columnas específicas: directorypath, numclips")
        print(f"  Guardando en: {experiment_splits_manifest.resolve()}\n")
        
        try:
            enrich_splits_manifest(
                base_splits_csv=str(base_splits_manifest),
                clips_manifest_csv=str(manifest_for_next_step),
                output_csv=str(experiment_splits_manifest)
            )
            manifest_for_next_step = experiment_splits_manifest
            print(f"\nManifest de splits listo para este experimento")
        except Exception as e:
            print(f"ERROR al enriquecer el manifest de splits: {e}")
            manifest_for_next_step = None
    else:
        manifest_for_next_step = None
else:
    print("Omitiendo división del dataset debido a error en pasos anteriores.")

Usando manifest: manifest_clips_original_len16_seg32_uniform.csv

Manifest base de splits ya existe: manifest_splitted_original.csv
  Este manifest define la distribución de splits para todos los experimentos
  Reutilizando splits existentes para garantizar consistencia

  Agregando columnas específicas: directorypath, numclips
  Guardando en: D:\Dataset\experiments\exp_50\results\tables\manifest_splits_clips_original_len16_seg32_uniform.csv


Manifest de splits enriquecido guardado: ..\experiments\exp_50\results\tables\manifest_splits_clips_original_len16_seg32_uniform.csv
Columnas: ['class', 'directoryname', 'directorypath', 'numclips', 'split']
Total de registros: 318

Manifest de splits listo para este experimento


## 4. Balancear el dataset de entrenamiento

In [60]:
print(f"Balanceo del dataset de TRAIN (Modo: {balance_mode})")

if balance_mode != 'none' and manifest_for_next_step:
    # El manifest balanceado se guarda en el directorio del experimento
    balanced_manifest_csv = manifests_subdir / f"manifest_balanced_{balance_mode}.csv"

    print(f"Manifest balanceado: {balanced_manifest_csv.resolve()}")
    
    # Verificar si ya existe el manifest balanceado
    if balanced_manifest_csv.exists():
        print(f"El manifest balanceado ya existe en: {balanced_manifest_csv.resolve()}")
        print("Reutilizando manifest balanceado existente.")
        manifest_for_next_step = balanced_manifest_csv
    else:
        print(f"Generando manifest balanceado en: {balanced_manifest_csv.resolve()}")
        
        try:
            if balance_mode == 'oversample':
                # Para oversample, los clips aumentados se guardan en directorio específico
                # El directorio fue generado automáticamente basándose en los parámetros
                print(f"\nDirectorio de clips aumentados (generado automáticamente):")
                print(f"  {oversample_clips_dir.resolve()}")
                print(f"\nLos clips aumentados son específicos para esta configuración:")
                print(f"  - clip_length: {clip_length}")
                print(f"  - max_segments_per_video: {max_segments}")
                print(f"  - overlapping: {overlapping}")
                print(f"  - stride: {stride if overlapping else 'N/A'}\n")
                
                %run ../src/data/balancer.py \
                    --input_csv {manifest_for_next_step} \
                    --output_csv {balanced_manifest_csv} \
                    --output_oversample_dir {oversample_clips_dir} \
                    --mode {balance_mode} \
                    --max_ratio {max_ratio} \
                    --seed {seed}
            else:
                # Para undersample, no se necesita directorio de salida
                %run ../src/data/balancer.py \
                    --input_csv {manifest_for_next_step} \
                    --output_csv {balanced_manifest_csv} \
                    --mode {balance_mode} \
                    --max_ratio {max_ratio} \
                    --seed {seed}

            manifest_for_next_step = balanced_manifest_csv
            print(f"Balanceo completado.")
        except Exception as e:
            print(f"ERROR al balancear el dataset: {e}")
            manifest_for_next_step = None
elif not manifest_for_next_step:
    print("Omitiendo balanceo del dataset debido a error en el paso anterior.")
else:
    print("Omitiendo balanceo del dataset (modo: none).")
    # Para modo 'none', copiar el manifest de splits al directorio del experimento
    final_manifest_csv = manifests_subdir / "manifest_final.csv"
    if not final_manifest_csv.exists() and manifest_for_next_step:
        import shutil
        shutil.copy(manifest_for_next_step, final_manifest_csv)
        print(f"Manifest final copiado a: {final_manifest_csv.resolve()}")
        manifest_for_next_step = final_manifest_csv

if manifest_for_next_step:
    print(f"\n{'='*60}")
    print(f"Preparación de datos completada para '{exp_name}'.")
    print(f"Manifest final: {manifest_for_next_step.resolve()}")
    print(f"{'='*60}\n")

Balanceo del dataset de TRAIN (Modo: undersample)
Manifest balanceado: D:\Dataset\experiments\exp_50\results\tables\manifest_balanced_undersample.csv
Generando manifest balanceado en: D:\Dataset\experiments\exp_50\results\tables\manifest_balanced_undersample.csv

=== Balanceo del conjunto de ENTRENAMIENTO ===
Clase minoritaria: 'Robbery' con 81 videos
Clase mayoritaria: 'Normal' con 122 videos
Ratio actual: 1.51
Aplicando undersampling. Videos de train seleccionados: 179
  - Robbery: 81
  - Normal: 98

Balanceo completado en modo 'undersample'.
Nuevo manifest CSV guardado en: D:\Dataset\experiments\exp_50\results\tables\manifest_balanced_undersample.csv
Distribución final de los videos de TRAIN:
  Clase 'Normal': 98 videos
  Clase 'Robbery': 81 videos
Balanceo completado.

Preparación de datos completada para 'exp_50'.
Manifest final: D:\Dataset\experiments\exp_50\results\tables\manifest_balanced_undersample.csv

